# MLOps Unit-II Practical: Monitoring Kidney Disease Classification

## Aim
Demonstrate the basics of monitoring a trained machine-learning model using the Kidney Disease dataset.

This practical monitors:
1. **Input data quality** — missing values and unseen categorical values.
2. **Inference performance** — how long batch prediction takes.
3. **Data drift** — whether numeric feature distributions change.
4. **Prediction quality** — compare model metrics on reference and artificially shifted inputs.
5. **Monitoring reports** — save monitoring results as CSV files.

### Important limitation
This is a **classroom simulation**, not a live hospital production system. The notebook uses a held-out test set as reference/production-like data and deliberately modifies one feature to demonstrate drift. The model metrics on the artificially shifted rows are only illustrative because the original labels may no longer be valid after changing feature values. Do not use this model for clinical decisions.


## Cell 1 — Load the Kidney Disease Dataset

### Theory
A monitoring workflow starts by loading the data that will be inspected. Consistent column names and trimmed text values reduce problems caused by accidental spaces, such as `"classification "` or `"ckd "`.

`Path` makes the file path easier to manage. The existence check gives a clear error if the CSV is not in the notebook's working directory.

### Before running
Place `kidney_disease_cleaned.csv` in the same folder as this notebook.


In [1]:
# ============================================================
# CELL 1 — IMPORTS AND LOAD THE KIDNEY DISEASE DATASET
# ============================================================

import pandas as pd             # Tables and CSV data manipulation
import numpy as np              # Numerical calculations (median, NaN checks, etc.)
import time                     # Measure model inference latency
import logging                  # Available for adding event logging in an extension

from pathlib import Path        # Portable path handling

# Expected dataset file in the notebook's current working directory.
DATA_FILE = Path("kidney_disease_cleaned.csv")

# Fail early with a clear message if the dataset cannot be found.
if not DATA_FILE.exists():
    raise FileNotFoundError(
        f"Dataset not found: {DATA_FILE.resolve()}"
    )

# Load the CSV into a pandas DataFrame.
df = pd.read_csv(DATA_FILE)

# Remove accidental leading/trailing spaces from column names.
df.columns = df.columns.str.strip()

# Clean leading/trailing spaces in text columns.
# This helps prevent values such as "ckd" and "ckd " from being treated
# as different categories. Missing values remain missing.
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].str.strip()

print("Dataset loaded successfully")
print("Dataset shape:", df.shape)  # (number of rows, number of columns)
print("Columns:", df.columns.tolist())

# Preview a few rows to confirm that the expected data was loaded.
display(df.head())


Dataset loaded successfully
Dataset shape: (400, 26)
Columns: ['id', 'age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hemo', 'pcv', 'wc', 'rc', 'htn', 'dm', 'cad', 'appet', 'pe', 'ane', 'classification']


,id,age,bp,sg,al,su,rbc,pc,pcc,ba,...,pcv,wc,rc,htn,dm,cad,appet,pe,ane,classification
0,0,48.0,80.0,1.020,1.0,0.0,NaN,normal,notpresent,notpresent,...,44,7800,5.2,yes,yes,no,good,no,no,ckd
1,1,7.0,50.0,1.020,4.0,0.0,NaN,normal,notpresent,notpresent,...,38,6000,NaN,no,no,no,good,no,no,ckd
2,2,62.0,80.0,1.010,2.0,3.0,normal,normal,notpresent,notpresent,...,31,7500,NaN,no,yes,no,poor,no,yes,ckd
3,3,48.0,70.0,1.005,4.0,0.0,normal,abnormal,present,notpresent,...,32,6700,3.9,yes,no,no,poor,yes,yes,ckd
4,4,51.0,80.0,1.010,2.0,0.0,normal,normal,notpresent,notpresent,...,35,7300,4.6,no,no,no,good,no,no,ckd


## Cell 2 — Inspect the Target and Missing Values

### Theory
The target column `classification` contains the class labels the model learns to predict. Its class distribution shows whether one class is much more common than another.

Missing-value counts are a simple **data-quality monitoring** signal. Missing data in a production input can indicate an upstream collection, form, or data-pipeline problem. `dropna=False` makes missing target labels visible in the class-count output.


In [2]:
# ============================================================
# CELL 2 — BASIC DATA-QUALITY INSPECTION
# ============================================================

# Check that the target column required for supervised learning exists.
print("Target column exists:", "classification" in df.columns)

# Show how many records belong to each target class.
# dropna=False ensures missing target labels are also counted.
print("\nTarget distribution:")
display(df["classification"].value_counts(dropna=False))

# Count missing values in every column.
# Sorting descending places the columns with the most missing values first.
print("\nMissing values per column:")
display(
    df.isna().sum()
      .sort_values(ascending=False)
      .to_frame("missing_count")
)


Target column exists: True

Target distribution:


classification
ckd       250
notckd    150
Name: count, dtype: int64


Missing values per column:


,missing_count
rbc,152
rc,130
wc,105
pot,88
sod,87
pcv,70
pc,65
hemo,52
su,49
sg,47


## Cell 3 — Normalize Target Labels

### Theory
Supervised classification requires a usable target label. This cell removes rows without a target and standardizes text labels so values such as `CKD`, `ckd.` and ` ckd ` do not accidentally become separate classes.

The `label_map` documents common CKD/non-CKD variants. Inspect the printed unique values: if your dataset contains other spellings, map them deliberately rather than silently assuming they are equivalent.


In [3]:
# ============================================================
# CELL 3 — CLEAN AND STANDARDIZE TARGET LABELS
# ============================================================

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

# Rows without a target label cannot be used for supervised training.
data = df.dropna(subset=["classification"]).copy()

# Convert labels to text and standardize case/whitespace.
# Examples: " CKD ", "ckd" and "CKD." become "ckd".
data["classification"] = (
    data["classification"]
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace(r"\s+", "", regex=True)
    .str.replace(r"\.$", "", regex=True)
)

# Map common spellings to consistent class labels.
# Extend this mapping if inspection reveals additional variants.
label_map = {
    "ckd": "ckd",
    "notckd": "notckd",
    "not_ckd": "notckd",
    "not ckd": "notckd"
}

data["classification"] = data["classification"].replace(label_map)

# Always inspect the classes printed here to ensure that the mapping
# has not accidentally merged or left inconsistent labels.
print("Target classes:", data["classification"].unique())


Target classes: ['ckd' 'notckd']


## Cell 4 — Separate Features and Target

### Theory
`X` contains model input features; `y` contains the target label. The `id` column is removed because it identifies a record and generally should not be treated as a medical predictor.

Numerical and categorical features require different preprocessing. Identifying their types lets the pipeline apply the correct transformation to each group.


In [4]:
# ============================================================
# CELL 4 — SEPARATE FEATURES AND TARGET
# ============================================================

# X contains the information used to predict the target.
# Remove both the target and the identifier column from the inputs.
# errors="ignore" prevents an error if the dataset has no "id" column.
X = data.drop(
    columns=["classification", "id"],
    errors="ignore"
)

# y contains the class label the model must learn to predict.
y = data["classification"]

# Identify numerical columns (for example, age or numeric lab measurements).
numeric_features = X.select_dtypes(
    include=["number"]
).columns.tolist()

# Treat all remaining columns as categorical/text-like inputs.
categorical_features = X.select_dtypes(
    exclude=["number"]
).columns.tolist()

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))
print("Samples:", len(X))


Numeric features: 11
Categorical features: 13
Samples: 400


## Cell 5 — Split Data and Train the Baseline Model

### Theory
The data is split into training and test sets. `stratify=y` attempts to preserve class proportions in both sets, while `random_state=42` makes the split reproducible.

The numerical pipeline imputes missing values using the median. The categorical pipeline imputes with the most frequent category and applies one-hot encoding. `handle_unknown="ignore"` prevents prediction from failing when a categorical value was not seen during training.

The `ColumnTransformer` applies each preprocessing pipeline to the appropriate columns. The scikit-learn `Pipeline` chains preprocessing and `RandomForestClassifier`, ensuring training and prediction use the same transformations.

`class_weight="balanced"` adjusts class weights inversely to class frequency; it can help when classes are imbalanced, but it does not guarantee better performance.


In [5]:
# ============================================================
# CELL 5 — SPLIT DATA, BUILD PREPROCESSING, TRAIN BASELINE MODEL
# ============================================================

# Hold out 20% of the data for evaluation.
# stratify=y helps retain class proportions in train and test sets.
# random_state=42 makes the split reproducible.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Numerical features: replace missing values with the training median.
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

# Categorical features:
# 1. Replace missing values with the most frequent training category.
# 2. One-hot encode categories as numeric indicator columns.
#    handle_unknown="ignore" allows categories not seen in training.
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Apply numerical and categorical preprocessing to their respective columns.
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

# Chain preprocessing and the classifier in one reusable scikit-learn Pipeline.
# class_weight="balanced" adjusts class weights based on class frequencies.
model = Pipeline([
    ("preprocessing", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        class_weight="balanced"
    ))
])

# Fit preprocessing and the Random Forest using training data only.
model.fit(X_train, y_train)

print("Baseline model trained successfully.")


Baseline model trained successfully.


## Cell 6 — Evaluate the Baseline Model

### Theory
Evaluation metrics summarize the model's performance on held-out test data:

- **Accuracy:** fraction of all predictions that are correct.
- **Macro precision:** precision is calculated for each class, then averaged equally across classes.
- **Macro recall:** recall is calculated for each class, then averaged equally.
- **Macro F1:** F1 is calculated per class and averaged equally.

Macro metrics are useful when class balance matters because each class contributes equally to the average. The classification report also gives per-class precision, recall, F1 and support. These are baseline values to compare against later; they are not production guarantees.


In [6]:
# ============================================================
# CELL 6 — BASELINE CLASSIFICATION METRICS
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

# Generate predictions for held-out records not used for fitting.
y_pred = model.predict(X_test)

# Calculate overall accuracy and macro-averaged precision, recall and F1.
# Macro averaging gives each class equal weight in the final average.
# zero_division=0 avoids warnings if a class receives no predicted samples.
baseline_metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "precision_macro": precision_score(
        y_test, y_pred, average="macro", zero_division=0
    ),
    "recall_macro": recall_score(
        y_test, y_pred, average="macro", zero_division=0
    ),
    "f1_macro": f1_score(
        y_test, y_pred, average="macro", zero_division=0
    )
}

# Put the metrics in a one-row table for easy display/export.
baseline_report = pd.DataFrame(
    [baseline_metrics],
    index=["Baseline"]
)

display(baseline_report.round(4))

# Show per-class precision, recall, F1-score and support.
print(classification_report(y_test, y_pred, zero_division=0))


,accuracy,precision_macro,recall_macro,f1_macro
Baseline,0.9875,0.9902,0.9833,0.9866


              precision    recall  f1-score   support

         ckd       0.98      1.00      0.99        50
      notckd       1.00      0.97      0.98        30

    accuracy                           0.99        80
   macro avg       0.99      0.98      0.99        80
weighted avg       0.99      0.99      0.99        80



## Cell 7 — Measure Batch Inference Latency

### Theory
Inference latency measures the time needed for the model to generate predictions. This cell measures the time to predict the entire test batch.

`time.perf_counter()` is a high-resolution timer. Multiplying elapsed seconds by 1000 converts the result to milliseconds. Average time per record is the batch time divided by the number of records.

This is a simple local benchmark; it does not include network, API, database, or cloud infrastructure latency.


In [7]:
# ============================================================
# CELL 7 — MEASURE BATCH INFERENCE LATENCY
# ============================================================

# Start a high-resolution timer immediately before prediction.
start = time.perf_counter()

# Simulate a batch prediction request using the held-out data.
production_predictions = model.predict(X_test)

# Convert elapsed time from seconds to milliseconds.
batch_latency_ms = (
    time.perf_counter() - start
) * 1000

print("Production batch size:", len(X_test))
print("Batch latency (ms):", round(batch_latency_ms, 3))

# This is a simple arithmetic average across records in this batch.
# It is not the same as measuring each record in an independent API request.
print(
    "Average time per record (ms):",
    round(batch_latency_ms / len(X_test), 3)
)


Production batch size: 80
Batch latency (ms): 17.003
Average time per record (ms): 0.213


## Cell 8 — Repeat Latency Measurements

### Theory
A single timing can be affected by warm-up, operating-system scheduling, and other temporary activity. Repeating the measurement ten times provides a small sample of observed latencies.

The **median** is less sensitive to unusually high measurements than the mean. The maximum is the slowest measurement observed in this small test, not a guaranteed worst-case latency.


In [8]:
# ============================================================
# CELL 8 — REPEAT LATENCY MEASUREMENTS
# ============================================================

# Repeat the same batch prediction ten times to observe timing variability.
latencies = []

for _ in range(10):
    start = time.perf_counter()
    model.predict(X_test)

    # Store each run's elapsed time in milliseconds.
    latencies.append(
        (time.perf_counter() - start) * 1000
    )

# Median is a robust summary for this small set of timings.
print("Median batch latency (ms):",
      round(np.median(latencies), 3))

# Maximum is the largest observed timing in these ten runs,
# not a guaranteed worst-case latency for a production service.
print("Maximum observed batch latency (ms):",
      round(max(latencies), 3))


Median batch latency (ms): 18.969
Maximum observed batch latency (ms): 31.174


## Cell 9 — Monitor Missing Input Values

### Theory
Production input quality can change over time. This report calculates the number and percentage of missing values for each input feature.

The notebook uses `X_test` as a stand-in for production-like data. In a real monitoring system, replace it with newly received records. The imputer allows the model to process missing values, but monitoring is still important because a sudden increase in missingness may signal a data-pipeline issue.


In [9]:
# ============================================================
# CELL 9 — MONITOR MISSING INPUT VALUES
# ============================================================

# In this practical, the test set stands in for production-like inputs.
# In a real deployment, replace it with newly arriving records.
production_data = X_test.copy()

# Calculate missing-value counts and percentages for each feature.
missing_report = pd.DataFrame({
    "missing_count": production_data.isna().sum(),
    "missing_percentage": (
        production_data.isna().mean() * 100
    ).round(2)
})

# Show the features with the highest percentage of missing values first.
display(
    missing_report.sort_values(
        "missing_percentage",
        ascending=False
    )
)


,missing_count,missing_percentage
rbc,25,31.25
rc,23,28.75
wc,17,21.25
pot,16,20.00
pc,16,20.00
sod,15,18.75
sg,14,17.50
su,13,16.25
al,12,15.00
pcv,10,12.50


## Cell 10 — Detect Unseen Categorical Values

### Theory
A category is **unseen** if it appears in the production-like data but did not appear in the training data for that feature.

The code compares sets of normalized string values. `OneHotEncoder(handle_unknown="ignore")` prevents an error for unseen values, but silently ignoring a new category does not mean the input is harmless. Investigate unexpected values and update data validation or retraining procedures when needed.

If nothing prints, no unseen categories were found by this check.


In [10]:
# ============================================================
# CELL 10 — CHECK FOR UNSEEN CATEGORICAL VALUES
# ============================================================

# Compare values observed during training with values in production-like data.
for col in categorical_features:

    # Use non-missing, trimmed string representations for comparison.
    train_values = set(
        X_train[col].dropna().astype(str).str.strip()
    )
    prod_values = set(
        production_data[col].dropna().astype(str).str.strip()
    )

    # Values present in production-like data but absent from training.
    unseen = prod_values - train_values

    # Report only columns where a new category was found.
    if unseen:
        print(f"{col}: unseen values = {unseen}")

# Note: OneHotEncoder(handle_unknown="ignore") prevents prediction failure,
# but unexpected new categories may still indicate a data-quality issue.


pcv: unseen values = {'20', '17', '16', '9'}
wc: unseen values = {'4900', '11500', '13600', '16300', '12300', '12700', '18900'}
rc: unseen values = {'2.5', '2.4'}


## Cell 11 — Simulate Feature Drift

### Theory
**Data drift** means the distribution of incoming input features changes relative to a reference distribution. This cell deliberately shifts the first numeric feature to create a visible example.

This is synthetic drift, created for teaching. It is not evidence that real production data has drifted. The reference and shifted datasets contain the same rows, except that one numeric feature is changed.


In [11]:
# ============================================================
# CELL 11 — SIMULATE NUMERICAL FEATURE DRIFT
# ============================================================

# Select the first numeric feature for a simple demonstration.
# Confirm numeric_features is not empty before using this approach.
feature = numeric_features[0]

# Use the same rows as a reference and a production-like copy.
reference_data = X_test.copy()
drifted_data = X_test.copy()

# Artificially shift one numeric feature to demonstrate drift detection.
reference_median = reference_data[feature].median()

if pd.notna(reference_median) and reference_median != 0:
    # Add half the absolute reference median to each value.
    drifted_data[feature] = (
        drifted_data[feature] + abs(reference_median) * 0.5
    )
else:
    # Fallback for a zero or missing median.
    drifted_data[feature] = drifted_data[feature] + 1

print("Feature modified:", feature)
print("Reference mean:",
      round(reference_data[feature].mean(), 3))
print("Simulated production mean:",
      round(drifted_data[feature].mean(), 3))

# This is synthetic data drift created for the practical, not a real alert.


Feature modified: age
Reference mean: 53.092
Simulated production mean: 81.092


## Cell 12 — Quantify Mean Shift

### Theory
For each numeric feature, the function compares the reference mean with the production-like mean. The shift score is:

`abs(production_mean - reference_mean) / reference_standard_deviation`

This expresses the mean difference in units of the reference standard deviation. A larger score indicates a larger mean shift relative to the reference variability.

This is a simple educational indicator, not a universal drift test or a validated alert threshold. It can be undefined when the reference standard deviation is zero or missing. For robust monitoring, consider feature-specific thresholds and statistical tests appropriate to the data.


In [12]:
# ============================================================
# CELL 12 — CALCULATE A SIMPLE MEAN-SHIFT SCORE
# ============================================================

def mean_shift_report(reference, production, columns):
    # One output row will be created for each numeric feature.
    rows = []

    for col in columns:
        # Reference and production-like means for this feature.
        ref_mean = reference[col].mean()
        prod_mean = production[col].mean()

        # Use reference variability to scale the size of the mean change.
        ref_std = reference[col].std()

        if pd.isna(ref_std) or ref_std == 0:
            # Cannot scale the difference if reference variation is unknown
            # or zero, so mark the score as not available.
            score = np.nan
        else:
            # Absolute mean difference in reference-standard-deviation units.
            score = abs(prod_mean - ref_mean) / ref_std

        rows.append({
            "feature": col,
            "reference_mean": ref_mean,
            "production_mean": prod_mean,
            "shift_score": score
        })

    # Sort largest scores first so the strongest shifts appear at the top.
    return pd.DataFrame(rows).sort_values(
        "shift_score",
        ascending=False,
        na_position="last"
    )

# Compare reference inputs with the artificially shifted copy.
drift_report = mean_shift_report(
    reference_data,
    drifted_data,
    numeric_features
)

display(drift_report.round(4))


,feature,reference_mean,production_mean,shift_score
0,age,53.0921,81.0921,1.5263
1,bp,77.0513,77.0513,0.0000
2,sg,1.0170,1.0170,0.0000
3,al,1.1029,1.1029,0.0000
4,su,0.3134,0.3134,0.0000
5,bgr,143.0541,143.0541,0.0000
6,bu,64.0513,64.0513,0.0000
7,sc,3.7094,3.7094,0.0000
8,sod,136.5923,136.5923,0.0000
9,pot,4.5188,4.5188,0.0000


## Cell 13 — Compare Predictions on Reference and Shifted Inputs

### Theory
The same trained model predicts on the reference and artificially shifted datasets. The resulting metrics are displayed side by side to show that changing inputs can change model predictions and measured metrics.

**Interpretation caution:** the shifted rows are made by changing a feature while reusing the original `y_test` labels. Those labels may not be valid for the modified feature values. Therefore, the second row is a controlled classroom demonstration, not a trustworthy estimate of real production accuracy. In production, performance monitoring requires verified labels when they become available.


In [ ]:
# ============================================================
# CELL 13 — COMPARE PREDICTIONS BEFORE AND AFTER ARTIFICIAL SHIFT
# ============================================================

# Predict using the original reference inputs.
reference_pred = model.predict(reference_data)

# Predict again after one numeric feature was artificially shifted.
drifted_pred = model.predict(drifted_data)

# Build a comparison table using the same held-out labels.
# IMPORTANT: labels are unchanged even though one feature is modified.
# Therefore, the shifted-data metrics are only a teaching illustration;
# they are not reliable production accuracy estimates.
comparison = pd.DataFrame([
    {
        "dataset": "Reference",
        "accuracy": accuracy_score(y_test, reference_pred),
        "precision_macro": precision_score(
            y_test, reference_pred,
            average="macro", zero_division=0
        ),
        "recall_macro": recall_score(
            y_test, reference_pred,
            average="macro", zero_division=0
        ),
        "f1_macro": f1_score(
            y_test, reference_pred,
            average="macro", zero_division=0
        )
    },
    {
        "dataset": "Artificially shifted",
        "accuracy": accuracy_score(y_test, drifted_pred),
        "precision_macro": precision_score(
            y_test, drifted_pred,
            average="macro", zero_division=0
        ),
        "recall_macro": recall_score(
            y_test, drifted_pred,
            average="macro", zero_division=0
        ),
        "f1_macro": f1_score(
            y_test, drifted_pred,
            average="macro", zero_division=0
        )
    }
]).set_index("dataset")

display(comparison.round(4))


## Cell 14 — Build a Monitoring Summary

### Theory
A monitoring summary brings several signals into one record: batch size, measured latency, total missing input values, baseline metrics, and the largest calculated mean-shift score.

`artificial_drift_test=True` records that this notebook deliberately simulated drift. It should not be interpreted as an automatically detected real-world incident.


In [ ]:
# ============================================================
# CELL 14 — SUMMARIZE MONITORING SIGNALS
# ============================================================

# Largest available numeric feature shift score from the drift report.
max_shift = drift_report["shift_score"].max()

# Collect the key signals into one dictionary/record for reporting.
monitoring_summary = {
    "production_records": len(production_data),
    "batch_latency_ms": round(batch_latency_ms, 3),

    # Count all missing cells across all input records and features.
    "missing_input_values": int(
        production_data.isna().sum().sum()
    ),

    # Baseline evaluation metrics from the held-out test set.
    "baseline_accuracy": baseline_metrics["accuracy"],
    "baseline_macro_f1": baseline_metrics["f1_macro"],

    # None is saved when no finite shift score can be calculated.
    "max_mean_shift_score": (
        round(float(max_shift), 4)
        if pd.notna(max_shift) else None
    ),

    # Explicitly record that the shift was deliberately simulated.
    "artificial_drift_test": True
}

# Display the summary vertically so each signal is easy to read.
display(pd.DataFrame([monitoring_summary]).T.rename(
    columns={0: "value"}
))


## Cell 15 — Save Monitoring Reports

### Theory
Monitoring results are more useful when saved so that they can be reviewed, compared over time, or consumed by a dashboard.

This cell creates a `monitoring_reports` directory if needed and saves:
- `kidney_monitoring_summary.csv` — one-row summary of the run.
- `kidney_drift_report.csv` — feature-level mean-shift calculations.
- `kidney_performance_comparison.csv` — reference versus artificially shifted metrics.

These are local CSV files, not a live monitoring service or dashboard.


In [ ]:
# ============================================================
# CELL 15 — SAVE MONITORING REPORTS AS CSV FILES
# ============================================================

# Create the output directory if it does not already exist.
output_dir = Path("monitoring_reports")
output_dir.mkdir(exist_ok=True)

# Save one-row summary with the key monitoring signals.
pd.DataFrame([monitoring_summary]).to_csv(
    output_dir / "kidney_monitoring_summary.csv",
    index=False
)

# Save the feature-level drift calculations.
drift_report.to_csv(
    output_dir / "kidney_drift_report.csv",
    index=False
)

# Save the metric comparison for reference and artificially shifted data.
comparison.to_csv(
    output_dir / "kidney_performance_comparison.csv"
)

print("Monitoring reports saved to:", output_dir.resolve())


## Final Review — What This Practical Demonstrates

### Monitoring flow

```text
Kidney Disease CSV
        ↓
Data Quality Checks
        ↓
Preprocessing + Trained Random Forest
        ↓
Baseline Evaluation
        ↓
Latency Measurement
        ↓
Missing / Unseen Input Checks
        ↓
Artificial Feature Drift
        ↓
Shift Score + Prediction Comparison
        ↓
CSV Monitoring Reports
```

### Key distinctions
- **Logging** records events and messages while software runs.
- **Monitoring** collects and reviews signals such as latency, missingness, drift and performance.
- **Data drift** is a change in input data distribution.
- **Model performance degradation** means predictive quality worsens; it cannot usually be confirmed without trustworthy labels.
- **This notebook** demonstrates local monitoring calculations. It does not yet integrate Prometheus, Grafana, Evidently, FastAPI, or a live production stream.

### Suggested student questions
1. Why remove the `id` column from model inputs?
2. Why are numerical and categorical values preprocessed differently?
3. What is the difference between batch latency and per-record average latency?
4. Why monitor missing and unseen values even if the model can process them?
5. What does the mean-shift score measure?
6. Why is artificially shifted test accuracy only illustrative in this notebook?
7. Which monitoring signals would you send to Prometheus/Grafana in a real deployment?
